# 02a Retrain EMA Checkpoints

Run this notebook on Colab A100 High-RAM. Colab packages are scoped to one runtime session, so this notebook replays the canonical base-dependency and Mamba bootstrap from Notebook 02 when necessary. A dependency or Torch pin may intentionally restart Colab; after reconnecting, rerun Notebook 02a from the first cell. It retrains the five Chapman folds and writes explicit EMA/raw checkpoint variants to a versioned Drive model run directory, not the historical `Drive/ECG-Ramba/model` directory. `fold*_final_ema.pt` at the pre-specified epoch is the manuscript OOF checkpoint; `fold*_best_ema.pt` is retained only as a validation-selected diagnostic. Do not continue to Notebook 02 until both variants pass verification and the current model-run pointer is written.


## Setup Repository And Drive Paths


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
MODEL_RUNS_DIR = DRIVE_ROOT / 'model_runs'
RETRAIN_EPOCHS = int(os.environ.get('ECG_RAMBA_RETRAIN_EPOCHS', '20'))
DEFAULT_MODEL_RUN_ID = f'ema_protocol_e{RETRAIN_EPOCHS}_v2'
MODEL_RUN_ID = os.environ.get('ECG_RAMBA_RETRAIN_RUN_ID', DEFAULT_MODEL_RUN_ID)
MODEL_DIR = Path(os.environ.get('ECG_RAMBA_MODEL_DIR', str(MODEL_RUNS_DIR / MODEL_RUN_ID))).expanduser()
LEGACY_MODEL_DIR = DRIVE_ROOT / 'model'
CURRENT_MODEL_POINTER = MODEL_RUNS_DIR / 'current_final_ema_model_dir.txt'
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path(os.environ.get('ECG_RAMBA_LOCAL_ROOT', '/content/ecg_ramba_runtime'))
LOCAL_EXTRACT_DIR = Path(os.environ.get('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman')))

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

from google.colab import drive
if not _drive_root_ready(DRIVE_ROOT):
    try:
        drive.mount(str(DRIVE_MOUNT))
    except Exception as exc:
        print(f'Drive mount initial attempt failed or was stale: {exc}')
        drive.mount(str(DRIVE_MOUNT), force_remount=True)
else:
    print('Drive root already visible:', DRIVE_ROOT)
if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')

def run(cmd, *, cwd=None, check=True, log_path=None):
    print('$', cmd, flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        return subprocess.run(cmd, shell=True, cwd=str(run_cwd), check=check)
    log_path = Path(log_path)
    if not log_path.is_absolute():
        log_path = run_cwd / log_path
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = proc.wait()
    print('Command log:', log_path)
    if check and return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if MODEL_DIR.resolve() == LEGACY_MODEL_DIR.resolve():
    raise RuntimeError('Refusing to retrain into the historical Drive/ECG-Ramba/model directory. Set ECG_RAMBA_MODEL_DIR to a model_runs subdirectory.')

if not REPO_DIR.exists():
    run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
run('git fetch origin', cwd=REPO_DIR)
run(f'git checkout {BRANCH}', cwd=REPO_DIR)
run(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR)
os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
_AUTHORITY_BOOTSTRAP_ALLOWED = False
_AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260723-v10'

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    requested_release_ref = _authority_os.environ.get(
        'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
    ).strip()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
            'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
    release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    def resolve_annotated_release_ref(release_ref):
        if not release_ref_pattern.fullmatch(release_ref):
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                'Moving branches are not accepted as code authority.'
            )
        object_type = git('cat-file', '-t', release_ref, check=False)
        if object_type.returncode or object_type.stdout.strip() != 'tag':
            raise RuntimeError(
                f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                'Run the current Notebook 00 after the reviewed release tag has been pushed.'
            )
        object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
        commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
        if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
            raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
        return commit, object_id

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
        print((fetch.stdout or '')[-2000:])

    manifest = None
    manifest_raw = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file():
            manifest_raw = manifest_path.read_text(encoding='utf-8')
            manifest = _authority_json.loads(manifest_raw)

    authority_update_needed = False
    update_reason = None
    release_ref = None
    release_ref_object_id = None

    if reset_requested:
        expected_commit = requested_commit
        authority_update_needed = True
        update_reason = 'explicit_environment_sha'
    elif manifest is None:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        release_ref = requested_release_ref
        expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
        authority_update_needed = True
        update_reason = 'initial_versioned_release_bootstrap'
    else:
        manifest_capability = manifest.get('capability')
        manifest_schema = int(manifest.get('schema_version', 0))
        legacy_manifest = (
            manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
        )
        current_manifest = (
            manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
        )
        if not legacy_manifest and not current_manifest:
            raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
        manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(manifest_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            if legacy_manifest:
                raise RuntimeError(
                    'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                    'to migrate it to the reviewed versioned release before running downstream notebooks.'
                )
            expected_commit = manifest_commit
            if requested_commit and requested_commit != expected_commit:
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                    'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                )
        else:
            release_ref = requested_release_ref
            release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            manifest_ref = str(manifest.get('authority_ref', '')).strip()
            manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
            if current_manifest and manifest_ref == release_ref:
                if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                    raise RuntimeError(
                        f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                        'Publish a new versioned tag instead of retagging an existing release.'
                    )
                expected_commit = manifest_commit
            else:
                expected_commit = release_commit
                authority_update_needed = True
                update_reason = (
                    'legacy_manifest_migration'
                    if legacy_manifest
                    else 'versioned_release_upgrade'
                )

    if not commit_pattern.fullmatch(expected_commit):
        raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if authority_update_needed:
        previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
        previous_ref = None if manifest is None else manifest.get('authority_ref')
        manifest = {
            'capability': 'canonical_versioned_git_release_v2',
            'schema_version': 2,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if release_ref is None
                else 'verified_annotated_versioned_release_tag'
            ),
            'authority_ref': release_ref,
            'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
            'authority_ref_object_id': release_ref_object_id,
            'update_reason': update_reason,
            'previous_git_commit': previous_commit,
            'previous_authority_ref': previous_ref,
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
            if concurrent_raw != manifest_raw and not reset_requested:
                concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                if not (
                    concurrent
                    and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                    and int(concurrent.get('schema_version', 0)) == 2
                    and str(concurrent.get('git_commit', '')).lower() == expected_commit
                    and concurrent.get('authority_ref') == release_ref
                    and str(concurrent.get('authority_ref_object_id', '')).lower()
                    == str(release_ref_object_id or '').lower()
                ):
                    raise RuntimeError('A concurrent runtime established a different code authority.')
                manifest = concurrent
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established/updated canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority release  :', manifest.get('authority_ref'))
    print('Authority manifest :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['ECG_RAMBA_MODEL_DIR'] = str(MODEL_DIR)
os.environ['ECG_RAMBA_EPOCHS'] = str(RETRAIN_EPOCHS)
os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_EXTRACT_DIR)
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '1'
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '1'

print('cwd       :', Path.cwd())
print('drive root:', DRIVE_ROOT)
print('model dir :', MODEL_DIR)
print('local root:', LOCAL_RUNTIME_ROOT)
print('extract dir:', LOCAL_EXTRACT_DIR)
print('epochs    :', RETRAIN_EPOCHS)
print('legacy dir:', LEGACY_MODEL_DIR)
print('pointer   :', CURRENT_MODEL_POINTER)
print('branch    :', subprocess.check_output('git branch --show-current', shell=True, text=True).strip())
print('commit    :', subprocess.check_output('git rev-parse HEAD', shell=True, text=True).strip())
print('git status:', subprocess.check_output('git status --short --branch', shell=True, text=True).strip())

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Runtime Bootstrap And Sanity Check


In [ ]:
import importlib
import json

installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
installer_nb = json.loads(installer_nb_path.read_text(encoding='utf-8'))

def canonical_installer_source(capability_marker, schema_marker):
    candidates = []
    for notebook_cell in installer_nb['cells']:
        if notebook_cell.get('cell_type') != 'code':
            continue
        installer_source = ''.join(notebook_cell.get('source', []))
        if capability_marker in installer_source and schema_marker in installer_source:
            candidates.append(installer_source)
    if len(candidates) != 1:
        raise RuntimeError(
            f'Canonical installer candidate_count={len(candidates)} for '
            f'capability={capability_marker!r} schema={schema_marker!r}; expected exactly one.'
        )
    return candidates[0]

print('Colab packages are runtime-scoped; validating the complete ECG-RAMBA environment for this session.')
base_installer = canonical_installer_source("BASE_INSTALLER_CAPABILITY = 'ecg_ramba_base_installer_v1'", 'BASE_INSTALLER_SCHEMA_VERSION = 1')
exec(compile(base_installer, str(installer_nb_path) + ':base-deps', 'exec'), globals(), globals())

import torch

print('Python:', sys.version)
print('Torch :', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Retraining requires a CUDA GPU. Select A100 High-RAM before running this notebook.')

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print('GPU RAM:', f'{gpu_mem_gb:.1f} GiB')
if gpu_mem_gb < 35:
    raise RuntimeError('Use A100 High-RAM for full five-fold retraining. T4/L4 can be too slow or memory-limited.')

required_modules = ['mamba_ssm', 'causal_conv1d']
module_errors = {}
for module in required_modules:
    try:
        importlib.import_module(module)
    except Exception as exc:
        module_errors[module] = repr(exc)

if module_errors:
    print('Mamba runtime requires installation/repair:', module_errors)
    model_installer = canonical_installer_source("MAMBA_INSTALLER_CAPABILITY = 'ecg_ramba_mamba_installer_v1'", 'MAMBA_INSTALLER_SCHEMA_VERSION = 1')
    exec(compile(model_installer, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
    importlib.invalidate_caches()

module_errors = {}
for module in required_modules:
    try:
        loaded = importlib.import_module(module)
        print('OK import:', module, getattr(loaded, '__file__', 'built-in'))
    except Exception as exc:
        module_errors[module] = repr(exc)

if module_errors:
    raise ImportError(
        'Runtime modules remain unavailable after bootstrap: ' + json.dumps(module_errors) + '. '
        'If Colab restarted while pinning base dependencies or Torch, rerun Notebook 02a from the first cell.'
    )


## Data And Cache Preflight


In [ ]:
from configs.config import PATHS, CONFIG_HASH, CONFIG

required = {
    'Chapman ZIP': Path(PATHS['zip_path']),
    'cache dir': Path(PATHS['cache_dir']),
    'clean cache': Path(PATHS['data_cache']),
    'extract dir': Path(PATHS['extract_dir']),
    'fold PCA cache': Path(PATHS['cache_dir']) / 'revision_pca_models',
    'fold feature cache': Path(PATHS['cache_dir']) / 'revision_feature_cache',
    'model dir': Path(PATHS['model_dir']),
}
for name, path in required.items():
    size = path.stat().st_size if path.is_file() else None
    size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
    if path.is_dir():
        npz_count = len(list(path.glob('*.npz')))
        joblib_count = len(list(path.glob('*.joblib')))
        print(f'{name:18s}: exists=True npz_count={npz_count} joblib_count={joblib_count} path={path}')
    else:
        print(f'{name:18s}: exists={path.exists()} size={size_gib} path={path}')
# Expected feature-cache names are fingerprinted from the cleaned cache subject order.
# If these files exist, train/evaluation will load them instead of regenerating MiniRocket/HRV.
def inspect_expected_feature_caches():
    import numpy as np
    from src.provenance import record_order_fingerprint

    clean_cache_path = Path(PATHS['data_cache'])
    if not clean_cache_path.is_file():
        print('Feature cache preflight: clean cache is missing, so exact MiniRocket/HRV fingerprint cannot be derived yet.')
        return

    with np.load(clean_cache_path, allow_pickle=True) as payload:
        subjects = payload['subjects']
        stored_fingerprint = (
            str(payload['record_order_fingerprint'].item())
            if 'record_order_fingerprint' in payload.files
            else None
        )
    computed_fingerprint = record_order_fingerprint(subjects)
    if stored_fingerprint and stored_fingerprint != computed_fingerprint:
        raise RuntimeError(
            'Clean cache fingerprint mismatch: '
            f'stored={stored_fingerprint} computed={computed_fingerprint}'
        )
    fingerprint = stored_fingerprint or computed_fingerprint
    n_records = len(subjects)
    cache_dir = Path(PATHS['cache_dir'])
    expected_feature_caches = {
        'expected RAW MiniRocket cache': cache_dir / f'rocket_raw_N{n_records}_C12_L5000_K10000_S42_R{fingerprint}.npz',
        'expected HRV36 cache': cache_dir / f'hrv36_N{n_records}_C12_L5000_R{fingerprint}.npz',
    }
    print('Feature cache fingerprint:', fingerprint)
    all_present = True
    for name, path in expected_feature_caches.items():
        exists = path.is_file()
        all_present = all_present and exists
        size = path.stat().st_size if exists else None
        size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
        print(f'{name:32s}: exists={exists} size={size_gib} path={path}')
    if all_present:
        print('Feature cache preflight: MiniRocket and HRV36 caches are present and should be reused.')
    else:
        print('Feature cache preflight: at least one feature cache is missing; that feature will be regenerated once and then cached.')

inspect_expected_feature_caches()

if not Path(PATHS['zip_path']).is_file():
    raise FileNotFoundError(f'Missing Chapman ZIP: {PATHS["zip_path"]}')

print('Use clean cache:', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('Save clean cache:', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('Local extract :', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('Config hash:', CONFIG_HASH)
print('Folds      :', CONFIG['n_folds'])
print('Epochs     :', CONFIG['epochs'])
print('Batch size :', CONFIG['batch_size'])
print('Aggregation:', CONFIG['aggregation_method'], 'q=', CONFIG['power_mean_q'])


## Run Five-Fold Retraining


In [ ]:
from pathlib import Path
import os
import shutil
import sys

FORENSIC_RETRAIN_STREAMING_LOG_CAPABILITY = 'stage_run_id_durable_stream_v1'
RUN_RETRAIN = os.environ.get('ECG_RAMBA_RUN_RETRAIN', '0') == '1'
RESUME_TRAINING = os.environ.get('ECG_RAMBA_RESUME_TRAINING', '1') == '1'
os.environ['ECG_RAMBA_RESUME_TRAINING'] = '1' if RESUME_TRAINING else '0'

log_path = Path('reports/revision/logs/retrain_best_ema_train.log')
durable_model_log_path = MODEL_DIR / 'retrain_best_ema_train.log'
training_command = f'"{sys.executable}" -u scripts/train.py'
print('Training log (latest local):', log_path)
print('Training logs (durable run history):', MIRROR_REVISION_ROOT / 'logs' / 'history' / 'retrain_best_ema_train')
print('Training log (model-run convenience copy):', durable_model_log_path)
existing_ckpts = sorted(MODEL_DIR.glob('fold*_*.pt'))
if RUN_RETRAIN and existing_ckpts and not RESUME_TRAINING:
    preview = ', '.join(path.name for path in existing_ckpts[:8])
    raise RuntimeError(
        f'Model run directory already contains {len(existing_ckpts)} checkpoint files: {MODEL_DIR}. '
        f'Preview: {preview}. Enable RESUME_TRAINING or select a new ECG_RAMBA_RETRAIN_RUN_ID.'
    )
if existing_ckpts and RESUME_TRAINING:
    print(f'Resume enabled: {len(existing_ckpts)} existing checkpoint files will be audited by scripts/train.py.')
print('$', training_command)

if RUN_RETRAIN:
    run(
        training_command,
        cwd=REPO_DIR,
        log_path=log_path,
    )
    if not log_path.is_file() or log_path.stat().st_size == 0:
        raise RuntimeError('Retraining completed without a non-empty latest-run log.')
    durable_model_log_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_model_log = durable_model_log_path.with_suffix(
        durable_model_log_path.suffix + '.partial'
    )
    shutil.copy2(log_path, temporary_model_log)
    os.replace(temporary_model_log, durable_model_log_path)
    print('Retraining finished:', log_path)
    print('Model-run log refreshed:', durable_model_log_path)
else:
    print(
        'RUN_RETRAIN=False; skipping training and leaving existing artifacts untouched. '
        'Set ECG_RAMBA_RUN_RETRAIN=1 only when retraining is intended.'
    )


## Verify Explicit Checkpoint Contract


In [ ]:
import json
import hashlib
import pandas as pd
import torch
import joblib
import numpy as np
from scripts.revision.common import save_json_atomic
FORENSIC_RETRAIN_CHECKPOINT_SPLIT_CONTRACT = 'folds_pkl_checkpoint_membership_v1'

model_dir = Path(os.environ['ECG_RAMBA_MODEL_DIR'])
expected = []
for fold in range(1, 6):
    expected.extend([
        model_dir / f'fold{fold}_best_ema.pt',
        model_dir / f'fold{fold}_best_raw.pt',
        model_dir / f'fold{fold}_final_ema.pt',
        model_dir / f'fold{fold}_final_raw.pt',
    ])
expected.extend([
    model_dir / 'folds.pkl',
    model_dir / 'training_log_epochs.csv',
    model_dir / 'cv_results_clean_core.csv',
    model_dir / 'fold_label_prevalence.csv',
    model_dir / 'fold_split_audit.json',
    model_dir / 'resume_integrity_audit.json',
    model_dir / 'resume_integrity_audit.csv',
])
missing = [str(path) for path in expected if not path.exists()]
if missing:
    raise FileNotFoundError('Missing retrain artifacts: ' + '; '.join(missing))

fold_contracts = joblib.load(model_dir / 'folds.pkl')
if len(fold_contracts) != 5:
    raise RuntimeError(f'Expected five reviewed folds, found {len(fold_contracts)}')
validation_membership = []
for fold, split in enumerate(fold_contracts, start=1):
    tr_idx = np.ascontiguousarray(np.asarray(split['tr_idx'], dtype=np.int64))
    va_idx = np.ascontiguousarray(np.asarray(split['va_idx'], dtype=np.int64))
    if len(np.intersect1d(tr_idx, va_idx)):
        raise RuntimeError(f'Fold {fold} train/validation membership overlaps')
    validation_membership.extend(va_idx.tolist())
if len(validation_membership) != len(set(validation_membership)):
    raise RuntimeError('Validation records overlap across folds')
training_source_paths = (
    'scripts/train.py', 'src/model.py', 'src/features.py', 'src/utils.py',
    'src/aggregation.py', 'src/training_data.py', 'src/data_loader.py', 'configs/config.py',
)
live_training_source_files = {
    relative: hashlib.sha256(Path(relative).read_bytes()).hexdigest()
    for relative in training_source_paths
}
live_training_source_sha = hashlib.sha256(
    json.dumps(live_training_source_files, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
live_training_source_bundle = {
    'schema_version': 1, 'files': live_training_source_files, 'sha256': live_training_source_sha,
}
rows = []
dataset_record_fingerprints = set()
for fold in range(1, 6):
    for checkpoint_kind in ['final_ema', 'best_ema']:
        ckpt = model_dir / f'fold{fold}_{checkpoint_kind}.pt'
        payload = torch.load(ckpt, map_location='cpu', weights_only=False)
        split = fold_contracts[fold - 1]
        expected_train_hash = hashlib.sha256(
            np.ascontiguousarray(np.asarray(split['tr_idx'], dtype=np.int64)).view(np.uint8)
        ).hexdigest()[:16]
        expected_val_hash = hashlib.sha256(
            np.ascontiguousarray(np.asarray(split['va_idx'], dtype=np.int64)).view(np.uint8)
        ).hexdigest()[:16]
        checkpoint_split = payload.get('split') or {}
        if checkpoint_split.get('train_index_hash') != expected_train_hash:
            raise RuntimeError(f'{ckpt} train split hash differs from folds.pkl')
        if checkpoint_split.get('val_index_hash') != expected_val_hash:
            raise RuntimeError(f'{ckpt} validation split hash differs from folds.pkl')
        if payload.get('weights_kind') != 'ema':
            raise RuntimeError(f'{ckpt} does not report weights_kind=ema')
        expected_rule = 'fixed_final_epoch' if checkpoint_kind == 'final_ema' else 'max_validation_f1_macro'
        if payload.get('selection_rule') != expected_rule:
            raise RuntimeError(f'{ckpt} selection_rule={payload.get("selection_rule")} expected={expected_rule}')
        if checkpoint_kind == 'final_ema' and int(payload.get('epoch', -1)) != RETRAIN_EPOCHS:
            raise RuntimeError(f'{ckpt} epoch does not match RETRAIN_EPOCHS={RETRAIN_EPOCHS}')
        dataset_record_fingerprint = payload.get('dataset_record_order_fingerprint')
        if not dataset_record_fingerprint:
            raise RuntimeError(f'{ckpt} lacks dataset_record_order_fingerprint')
        dataset_record_fingerprints.add(str(dataset_record_fingerprint))
        h = hashlib.sha256(ckpt.read_bytes()).hexdigest()
        rows.append({
            'fold': fold,
            'checkpoint_kind': checkpoint_kind,
            'manuscript_role': 'primary_fixed_epoch' if checkpoint_kind == 'final_ema' else 'diagnostic_validation_selected',
            'path': str(ckpt),
            'sha256': h,
            'epoch': payload.get('epoch'),
            'f1_macro': payload.get('f1_macro'),
            'weights_kind': payload.get('weights_kind'),
            'metrics_weights_kind': payload.get('metrics_weights_kind'),
            'selection_rule': payload.get('selection_rule'),
            'config_hash': payload.get('config_hash'),
            'dataset_record_order_fingerprint': dataset_record_fingerprint,
            'aggregation': payload.get('aggregation'),
            'pca_explained_variance': payload.get('pca_explained_variance'),
            'training_protocol': payload.get('training_protocol'),
        })

if len(dataset_record_fingerprints) != 1:
    raise RuntimeError(f'Checkpoint dataset fingerprints differ: {sorted(dataset_record_fingerprints)}')

manifest_payload = {
    'model_run_id': MODEL_RUN_ID,
    'model_dir': str(model_dir),
    'legacy_model_dir': str(LEGACY_MODEL_DIR),
    'current_model_pointer': str(CURRENT_MODEL_POINTER),
    'durable_training_log': str(MODEL_DIR / 'retrain_best_ema_train.log'),
    'overwrite_legacy_model_dir': False,
    'epochs': RETRAIN_EPOCHS,
    'dataset_record_order_fingerprint': next(iter(dataset_record_fingerprints)),
    'checkpoints': rows,
}
manifest_path = Path('reports/revision/manifests/retrain_best_ema_checkpoint_manifest.json')
manifest_path.parent.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame(rows))

cv_path = model_dir / 'cv_results_clean_core.csv'
train_log_path = model_dir / 'training_log_epochs.csv'
prevalence_path = model_dir / 'fold_label_prevalence.csv'
resume_audit_path = model_dir / 'resume_integrity_audit.json'
resume_audit_csv_path = model_dir / 'resume_integrity_audit.csv'
cv_results = pd.read_csv(cv_path)
train_log = pd.read_csv(train_log_path)
resume_audit = json.loads(resume_audit_path.read_text(encoding='utf-8'))
resume_rows = pd.read_csv(resume_audit_csv_path)
if sorted(resume_audit.get('completed_folds_reused', [])) != sorted(resume_rows.loc[resume_rows['complete_for_resume_skip'].astype(bool), 'fold'].astype(int).tolist()):
    raise RuntimeError('Resume integrity audit JSON/CSV disagree about reused folds.')
if len(train_log) != 5 * RETRAIN_EPOCHS:
    raise RuntimeError(f'Expected {5 * RETRAIN_EPOCHS} epoch rows, found {len(train_log)}')
for fold in range(1, 6):
    fold_log = train_log[train_log['fold'] == fold]
    best_rows = fold_log[fold_log['is_best_epoch'].astype(bool)]
    if len(best_rows) != 1 or best_rows.iloc[0]['validation_weights_kind'] != 'ema':
        raise RuntimeError(f'Fold {fold} must have exactly one EMA best-epoch row; found {len(best_rows)}')

print('CV results:', cv_path)
display(cv_results)
print('Resume integrity audit:', resume_audit_path)
print('Reused folds:', resume_audit.get('completed_folds_reused'))
display(resume_rows)
print('Training log tail:')
display(train_log.tail(10))
print('Fold prevalence audit:', prevalence_path)
display(pd.read_csv(prevalence_path).head(10))

near_boundary = cv_results['best_ema_epoch'] >= (RETRAIN_EPOCHS - 1)
n_near_boundary = int(near_boundary.sum())
print(f'Best epoch within final two epochs: {n_near_boundary}/5 folds')
if RETRAIN_EPOCHS == 20 and n_near_boundary >= 3:
    print('Diagnostic recommendation: a separate e30 sensitivity run may test convergence, but do not replace the pre-specified e20 manuscript OOF by choosing the better result on these same folds.')
else:
    print('No automatic evidence that the epoch horizon is too short; decide from OOF and fold-level curves.')

# Promote this run only after all checkpoint/log/resume assertions above pass.
manifest_payload['training_source_bundle'] = live_training_source_bundle
save_json_atomic(manifest_path, manifest_payload)
CURRENT_MODEL_POINTER.parent.mkdir(parents=True, exist_ok=True)
pointer_tmp = CURRENT_MODEL_POINTER.with_name(CURRENT_MODEL_POINTER.name + '.partial.' + os.urandom(8).hex())
with pointer_tmp.open('w', encoding='utf-8') as handle:
    handle.write(str(model_dir) + '\n')
    handle.flush()
    os.fsync(handle.fileno())
os.replace(pointer_tmp, CURRENT_MODEL_POINTER)
print('Wrote:', manifest_path)
print('Promoted current model pointer:', CURRENT_MODEL_POINTER, '->', model_dir)


## Publish Retraining Provenance

Checkpoints and the live training log already reside in `model_runs`; this cell verifies and mirrors the revision manifest/log.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/retrain_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{MIRROR_REVISION_ROOT}"',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/retrain_mirror_publish.log',
)
print('Durable model run:', MODEL_DIR)
print('Canonical revision mirror:', MIRROR_REVISION_ROOT)
print('Retraining provenance publish complete.')
